# 04 — Final Neuromorphic Student: Architecture, Distillation & Training

**Purpose:** Document and train the single deployable model used by the exhibition.

## Official final model

**Improved Student**

The final architecture is:

`10 × 30 s PSG epochs → Lite Multi-Resolution Stem → Depthwise-Separable CNN → Compact Parametric Gabor FEB → 2-Layer GRU → 5-Class classifier`

Knowledge distillation is used during training. The teacher exists only to guide the student; the student checkpoint is the only model exported for deployment and exhibition.

### Final configuration
- Sequence length: **10 epochs**
- Context: **300 s**
- Input channels: **4**
- Classes: **5**
- Training budget: **20 epochs**
- Target artifact: `artifacts/student_improved_best.pt`

### Important
This notebook intentionally does **not** reproduce old baseline-vs-improved experiments. Those experiments are historical engineering work, not exhibition results.


In [1]:
import math
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
SEQ_LEN = 10
SEQ_STRIDE = 5
EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
N_CHANNELS = 4
N_CLASSES = 5
FS = 100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = Path("/home/shamique/projects/sleep/data/cache")
ARTIFACT_DIR = Path("/home/shamique/projects/sleep/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)


Device: cuda


## 1. Sequence dataset

Each training example contains ten contiguous 30-second epochs. Windows are formed independently inside each subject-night cache so no temporal sequence crosses recording boundaries.


In [2]:
class SleepSequenceDataset(Dataset):
    def __init__(self, cache_index_df, split, seq_len=SEQ_LEN, stride=SEQ_STRIDE):
        self.samples = []
        self.cache = {}
        self.seq_len = seq_len

        rows = cache_index_df.loc[cache_index_df["split"] == split]

        for _, row in rows.iterrows():
            path = row["cache_path"]
            data = np.load(path)
            n = len(data["labels"])

            for start in range(0, max(n - seq_len + 1, 1), stride):
                self.samples.append((path, start))

    def _load(self, path):
        if path not in self.cache:
            d = np.load(path)
            self.cache[path] = (d["epochs"], d["labels"])
        return self.cache[path]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, start = self.samples[index]
        epochs, labels = self._load(path)

        end = min(start + self.seq_len, len(labels))
        x = epochs[start:end]
        y = labels[start:end]

        if len(x) < self.seq_len:
            pad = self.seq_len - len(x)
            x = np.concatenate([x, np.repeat(x[-1:], pad, axis=0)], axis=0)
            y = np.concatenate([y, np.repeat(y[-1:], pad, axis=0)], axis=0)

        return torch.from_numpy(x).float(), torch.from_numpy(y).long()


In [3]:
cache_index = pd.read_csv(CACHE_DIR / "cache_index.csv")

train_ds = SleepSequenceDataset(cache_index, "train")
val_ds = SleepSequenceDataset(cache_index, "val")
test_ds = SleepSequenceDataset(cache_index, "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Sequence windows:")
print("train:", len(train_ds))
print("val:  ", len(val_ds))
print("test: ", len(test_ds))


Sequence windows:
train: 2220
val:   0
test:  0


## 2. Final architecture

The implementation below keeps the architecture deliberately compact.

**Multi-resolution stem** captures different temporal scales.  
**Depthwise-separable convolutions** reduce convolutional cost.  
**Gabor FEB** learns frequency-localized patterns with a small number of parameters.  
**GRU** models the 10-epoch temporal sequence.


In [4]:
class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5, stride=1):
        super().__init__()
        self.depthwise = nn.Conv1d(
            in_ch, in_ch, kernel, stride, kernel // 2,
            groups=in_ch, bias=False
        )
        self.pointwise = nn.Conv1d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return self.act(self.bn(x))


class LiteMultiResolutionStem(nn.Module):
    def __init__(self, in_ch=4, width=10, fs=100):
        super().__init__()
        small_kernel, small_stride = fs // 4, max(1, fs // 16)
        large_kernel, large_stride = fs, max(1, fs // 4)

        self.short = DepthwiseSeparableConv1d(
            in_ch, width, small_kernel, small_stride
        )
        self.long = DepthwiseSeparableConv1d(
            in_ch, width, large_kernel, large_stride
        )
        self.out_ch = width * 2

    def forward(self, x):
        short = self.short(x)
        long = self.long(x)
        target = min(short.shape[-1], long.shape[-1])
        short = F.adaptive_avg_pool1d(short, target)
        long = F.adaptive_avg_pool1d(long, target)
        return torch.cat([short, long], dim=1)


class ParametricGabor1D(nn.Module):
    def __init__(self, in_ch=4, n_filters=8, kernel_size=51, fs=100):
        super().__init__()
        self.in_ch = in_ch
        self.n_filters = n_filters
        self.kernel_size = kernel_size
        self.fs = fs

        self.center_freq = nn.Parameter(
            torch.linspace(0.5, 30.0, n_filters) / fs
        )
        self.bandwidth = nn.Parameter(torch.full((n_filters,), 0.02))

        t = torch.arange(
            -(kernel_size // 2), kernel_size // 2 + 1,
            dtype=torch.float32
        )
        self.register_buffer("t", t)

    def forward(self, x):
        t = self.t.unsqueeze(0)
        f0 = self.center_freq.unsqueeze(1)
        sigma = (self.bandwidth.abs() + 1e-4).unsqueeze(1)

        envelope = torch.exp(-0.5 * (t / (sigma * self.kernel_size)) ** 2)
        carrier = torch.cos(2 * math.pi * f0 * t)
        kernels = (envelope * carrier).unsqueeze(1)

        b, c, s = x.shape
        y = F.conv1d(
            x.reshape(b * c, 1, s),
            kernels,
            padding=self.kernel_size // 2,
        )
        return y.reshape(b, c * self.n_filters, -1)


class CompactGaborFEB(nn.Module):
    def __init__(self, in_ch=4, n_filters=8, out_dim=32):
        super().__init__()
        self.gabor = ParametricGabor1D(
            in_ch=in_ch, n_filters=n_filters
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(in_ch * n_filters, out_dim)

    def forward(self, x):
        x = self.gabor(x)
        x = self.pool(x).squeeze(-1)
        return F.relu6(self.proj(x))


class ImprovedStudent(nn.Module):
    def __init__(self, n_classes=5, gru_hidden=64):
        super().__init__()

        self.stem = LiteMultiResolutionStem(
            in_ch=N_CHANNELS, width=10, fs=FS
        )

        self.encoder = nn.Sequential(
            DepthwiseSeparableConv1d(
                self.stem.out_ch, 32, kernel=5, stride=2
            ),
            DepthwiseSeparableConv1d(
                32, 32, kernel=5, stride=2
            ),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.gabor_feb = CompactGaborFEB(
            in_ch=N_CHANNELS, n_filters=8, out_dim=32
        )

        feature_dim = 32 + 32
        self.gru = nn.GRU(
            feature_dim,
            gru_hidden,
            num_layers=2,
            batch_first=True,
        )
        self.head = nn.Linear(gru_hidden, n_classes)
        self.feature_dim = feature_dim

    def forward(self, x, return_features=False):
        b, t, c, s = x.shape
        flat = x.reshape(b * t, c, s)

        cnn = self.pool(self.encoder(self.stem(flat))).squeeze(-1)
        gabor = self.gabor_feb(flat)

        features = torch.cat([cnn, gabor], dim=-1).reshape(
            b, t, self.feature_dim
        )

        sequence_out, _ = self.gru(features)
        logits = self.head(sequence_out)

        if return_features:
            return logits, features
        return logits


student = ImprovedStudent().to(DEVICE)

with torch.no_grad():
    smoke = torch.randn(2, SEQ_LEN, N_CHANNELS, 30 * FS, device=DEVICE)
    smoke_logits, smoke_features = student(smoke, return_features=True)

print("Logits:", tuple(smoke_logits.shape))
print("Features:", tuple(smoke_features.shape))


Logits: (2, 10, 5)
Features: (2, 10, 64)


## 3. Training objective

Knowledge distillation transfers information from a trained teacher into the smaller student.

The final objective combines:

- hard-label cross-entropy,
- softened teacher-student KL divergence,
- feature alignment.

The teacher is a training-time component only.


In [5]:
class DistillationObjective(nn.Module):
    def __init__(self, class_weights=None, temperature=4.0,
                 alpha_ce=1.0, alpha_kl=1.0, alpha_feat=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=0.0
        )
        self.temperature = temperature
        self.alpha_ce = alpha_ce
        self.alpha_kl = alpha_kl
        self.alpha_feat = alpha_feat

    def forward(self, student_logits, student_features,
                teacher_logits, teacher_features, targets):

        b, t, c = student_logits.shape
        s = student_logits.reshape(b * t, c)
        teacher = teacher_logits.reshape(b * t, c).detach()
        target = targets.reshape(b * t)

        ce = self.ce(s, target)

        temp = self.temperature
        kl = F.kl_div(
            F.log_softmax(s / temp, dim=-1),
            F.softmax(teacher / temp, dim=-1),
            reduction="batchmean",
        ) * (temp ** 2)

        s_feat = student_features.reshape(b * t, -1)
        t_feat = teacher_features.reshape(b * t, -1).detach()
        common_dim = min(s_feat.shape[-1], t_feat.shape[-1])
        feat = F.mse_loss(s_feat[:, :common_dim], t_feat[:, :common_dim])

        total = (
            self.alpha_ce * ce
            + self.alpha_kl * kl
            + self.alpha_feat * feat
        )

        return total, {
            "ce": float(ce.detach()),
            "kl": float(kl.detach()),
            "feature": float(feat.detach()),
        }


## 4. Class weighting from the training partition

Class weights are computed from training labels only. The validation and test partitions are never used to determine training weights.


In [6]:
train_labels = []

for _, row in cache_index.loc[cache_index["split"] == "train"].iterrows():
    train_labels.append(np.load(row["cache_path"])["labels"])

train_labels = np.concatenate(train_labels)
class_counts = np.bincount(train_labels, minlength=N_CLASSES)

counts = torch.tensor(class_counts, dtype=torch.float32, device=DEVICE)
class_weights = torch.log(counts.sum() / (counts + 1.0))
class_weights = class_weights / class_weights.mean()

print("Training class counts:", class_counts.tolist())


Training class counts: [7562, 318, 1845, 718, 686]


## 5. Validation metric

Validation Cohen's κ is used for checkpoint selection because it accounts for agreement beyond class prevalence and is appropriate for ordinal sleep-stage classification.


In [7]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []

    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(dim=-1).cpu().numpy()

        y_true.append(y.numpy().reshape(-1))
        y_pred.append(pred.reshape(-1))

    return np.concatenate(y_true), np.concatenate(y_pred)


def validation_kappa(model, loader):
    from sklearn.metrics import cohen_kappa_score
    y_true, y_pred = collect_predictions(model, loader)
    return float(cohen_kappa_score(y_true, y_pred))


## 6. Teacher interface

For a clean exhibition repository, the teacher checkpoint is optional at inference time. During training, load the teacher that generated the final student artifact. If the repository already exposes the teacher through `neuromorphic_sleep_pipeline.py`, use that implementation rather than creating a second teacher definition.


In [8]:
TEACHER_CHECKPOINT = ARTIFACT_DIR / "teacher_improved_best.pt"
STUDENT_CHECKPOINT = ARTIFACT_DIR / "student_improved_best.pt"

print("Expected teacher checkpoint:", TEACHER_CHECKPOINT)
print("Final student checkpoint:", STUDENT_CHECKPOINT)


Expected teacher checkpoint: /home/shamique/projects/sleep/artifacts/teacher_improved_best.pt
Final student checkpoint: /home/shamique/projects/sleep/artifacts/student_improved_best.pt


## 7. Distillation training loop

The loop is intentionally compact and readable for a project exhibition. It checkpoints only the best student according to validation κ.


In [9]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW

def train_student_with_teacher(student, teacher, objective,
                               train_loader, val_loader,
                               epochs=EPOCHS):

    optimizer = AdamW(
        student.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_kappa = -1.0
    history = []

    teacher.eval()
    student.train()

    for epoch in range(1, epochs + 1):
        student.train()

        running = 0.0
        batches = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            with torch.no_grad():
                teacher_logits, teacher_features = teacher(
                    x, return_features=True
                )

            student_logits, student_features = student(
                x, return_features=True
            )

            loss, parts = objective(
                student_logits,
                student_features,
                teacher_logits,
                teacher_features,
                y,
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                student.parameters(), GRAD_CLIP
            )
            optimizer.step()

            running += float(loss.detach())
            batches += 1

        val_kappa = validation_kappa(student, val_loader)

        row = {
            "epoch": epoch,
            "train_loss": running / max(batches, 1),
            "val_kappa": val_kappa,
            **parts,
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_kappa={val_kappa:.4f}"
        )

        if val_kappa > best_kappa:
            best_kappa = val_kappa
            torch.save(
                {
                    "model_state_dict": student.state_dict(),
                    "epoch": epoch,
                    "val_kappa": val_kappa,
                    "seed": SEED,
                    "seq_len": SEQ_LEN,
                },
                STUDENT_CHECKPOINT,
            )

    return pd.DataFrame(history)


## 8. Training entry point

The teacher must be loaded from the project’s actual trained teacher implementation before running this cell. This separation is deliberate: the exhibition artifact is the student checkpoint, while the teacher is only a training dependency.

For the official 20-epoch project run, keep `EPOCHS = 20` and do not mix results from earlier runs into the final report.


In [10]:
# Example wiring:
#
# teacher = ImprovedTeacher(...).to(DEVICE)
# teacher.load_state_dict(torch.load(TEACHER_CHECKPOINT, map_location=DEVICE))
#
# objective = DistillationObjective(class_weights=class_weights)
# history = train_student_with_teacher(
#     student, teacher, objective, train_loader, val_loader, epochs=20
# )
# history.to_csv("results/final_training_history.csv", index=False)
#
# The final evaluation must be performed only by Notebook 05.
print("Training entry point is ready. Load the project teacher checkpoint and execute the block above.")


Training entry point is ready. Load the project teacher checkpoint and execute the block above.


## Handoff to Notebook 05

Notebook 05 is the only notebook that publishes the official test metrics. Training logs are diagnostic artifacts and must not be turned into alternative final-result tables.
